# Lab 9- Deep Learning Model

This lab is meant to get you started in using Keras to design Deep Neural Networks. The goal here is to simply repeat your previous lab, but with DNNs.

Lecture 21 is your template: it reads the same data, builds, compiles and trains a network, plots its learning curves, and makes a ROC curve. Copy its cells and change the numbers.

Let's start with reading the data, like before. It reads the full file you downloaded in Lab 7 (5 million rows, about 760 MB in memory once loaded). If your computer struggles, read `../Lab.7/SUSY-small.csv` instead (the first 500,000 rows) and divide the sample sizes below by ten.

`RawNames` are the 8 quantities measured directly; `FeatureNames` are the 10 derived from them, as in Lab 8. You will need both in Exercise 2.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

filename="../Lab.7/Lab7_Submission/SUSY.csv"
VarNames=["signal", "l_1_pT", "l_1_eta","l_1_phi", "l_2_pT", "l_2_eta", "l_2_phi", "MET", "MET_phi", "MET_rel", "axial_MET", "M_R", "M_TR_2", "R", "MT2", "S_R", "M_Delta_R", "dPhi_r_b", "cos_theta_r1"]
RawNames=["l_1_pT", "l_1_eta","l_1_phi", "l_2_pT", "l_2_eta", "l_2_phi", "MET", "MET_phi"]
FeatureNames=[v for v in VarNames[1:] if v not in RawNames]

df = pd.read_csv(filename, dtype='float64', names=VarNames)

Now let's define the samples. DNNs take long to train, so to start we use only the first 500,000 of the 5 million events for training. Once you get everything working, make the final version of your plots with more.

We keep **three** samples, because they do different jobs:

* **training**: what `fit` learns from;
* **validation**: what you watch while you work (pass it to `fit` as `validation_data`), and what you use to choose between models;
* **test**: used only at the end, to report the performance of the models you chose.

Keras works on NumPy arrays, so after doing all of the nice manipulation that pandas enables, we convert the DataFrames to NumPy arrays.

In [2]:
N_Train=500000
N_Val=50000
N_Test=50000

Train_Sample=df[:N_Train]
Val_Sample=df[N_Train:N_Train+N_Val]
Test_Sample=df[N_Train+N_Val:N_Train+N_Val+N_Test]

X_Train=np.array(Train_Sample[VarNames[1:]])
y_Train=np.array(Train_Sample["signal"])

X_Val=np.array(Val_Sample[VarNames[1:]])
y_Val=np.array(Val_Sample["signal"])

X_Test=np.array(Test_Sample[VarNames[1:]])
y_Test=np.array(Test_Sample["signal"])

print(X_Train.shape, X_Val.shape, X_Test.shape)

(500000, 18) (50000, 18) (50000, 18)


## Exercise 1

You will need to create several models and make sure they are properly trained. Write a function that takes this history and plots the values versus epoch. For every model that you train in the remainder of this lab, assess:

* Has your model's performance plateaued? If not train for more epochs. 
* Compare the performance on the training and validation samples. Are you over training?

In [3]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import roc_curve, auc


def plot_learning_curves(history, model_name="DNN Model"):
    hist = history.history
    epochs = range(1, len(hist["loss"]) + 1)
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
  
    axes[0].plot(epochs, hist["loss"], "b-o", label="Training Loss")
    axes[0].plot(epochs, hist["val_loss"], "r--s", label="Validation Loss")
    axes[0].set_title(f"{model_name}: Loss vs. Epoch")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")
    axes[0].legend()
    axes[0].grid(alpha=0.3)
    
    
    acc_key = "accuracy" if "accuracy" in hist else "acc"
    val_acc_key = "val_accuracy" if "val_accuracy" in hist else "val_acc"
    if acc_key in hist:
        axes[1].plot(epochs, hist[acc_key], "b-o", label="Training Accuracy")
        axes[1].plot(epochs, hist[val_acc_key], "r--s", label="Validation Accuracy")
        axes[1].set_title(f"{model_name}: Accuracy vs. Epoch")
        axes[1].set_xlabel("Epoch")
        axes[1].set_ylabel("Accuracy")
        axes[1].legend()
        axes[1].grid(alpha=0.3)
        
    plt.tight_layout()
    plt.show()

ModuleNotFoundError: No module named 'tensorflow'

## Exercise 2

Following the original paper (see Lab 7), make a comparison of the performance (using ROC curves and AUC) between models trained with the raw quantities (`RawNames`), the derived features (`FeatureNames`), and both. Note that the first layer's input shape changes with the number of inputs.

In [ ]:

X_Train_Raw = np.array(Train_Sample[RawNames])
X_Val_Raw   = np.array(Val_Sample[RawNames])
X_Test_Raw  = np.array(Test_Sample[RawNames])

X_Train_Feat = np.array(Train_Sample[FeatureNames])
X_Val_Feat   = np.array(Val_Sample[FeatureNames])
X_Test_Feat  = np.array(Test_Sample[FeatureNames])

def build_baseline_dnn(input_dim):
    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(64, activation="relu"),
        layers.Dense(64, activation="relu"),
        layers.Dense(1, activation="sigmoid")
    ])
    model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    return model

early_stop = keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)

feature_sets = {
    "Raw Quantities (8)": (X_Train_Raw, X_Val_Raw, X_Test_Raw),
    "Derived Features (10)": (X_Train_Feat, X_Val_Feat, X_Test_Feat),
    "Both / All Features (18)": (X_Train, X_Val, X_Test)
}

ex2_models = {}
ex2_roc = {}

for label, (x_tr, x_v, x_te) in feature_sets.items():
    print(f"\n--- Training DNN on {label} ---")
    model = build_baseline_dnn(x_tr.shape[1])
    history = model.fit(
        x_tr, y_Train,
        validation_data=(x_v, y_Val),
        epochs=15,
        batch_size=1024,
        callbacks=[early_stop],
        verbose=1
    )
    plot_learning_curves(history, model_name=f"DNN ({label})")
    
    preds = model.predict(x_v, batch_size=2048).ravel()
    fpr, tpr, _ = roc_curve(y_Val, preds)
    ex2_roc[label] = (fpr, tpr, auc(fpr, tpr))
    ex2_models[label] = model


plt.figure(figsize=(8, 6))
for label, (fpr, tpr, roc_auc) in ex2_roc.items():
    plt.plot(fpr, tpr, lw=2, label=f"{label} (Val AUC = {roc_auc:.4f})")
plt.plot([0, 1], [0, 1], "k--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Exercise 2: DNN ROC Comparison (Raw vs. Derived vs. Both)")
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.show()

## Exercise 3

Design and implement at least 3 different DNN models. Train them and compare performance. You may try different architectures, loss functions, and optimizers to see if there is an effect.

In [ ]:

dnn_1 = keras.Sequential([
    layers.Input(shape=(18,)),
    layers.Dense(32, activation="tanh"),
    layers.Dense(1, activation="sigmoid")
], name="Model_1_Shallow_Tanh_SGD")
dnn_1.compile(optimizer=keras.optimizers.SGD(learning_rate=0.05, momentum=0.9),
              loss="binary_crossentropy", metrics=["accuracy"])


dnn_2 = keras.Sequential([
    layers.Input(shape=(18,)),
    layers.Dense(128, activation="relu"),
    layers.BatchNormalization(),
    layers.Dropout(0.2),
    layers.Dense(128, activation="relu"),
    layers.BatchNormalization(),
    layers.Dropout(0.2),
    layers.Dense(64, activation="relu"),
    layers.Dense(1, activation="sigmoid")
], name="Model_2_Deep_BN_Adam")
dnn_2.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3),
              loss="binary_crossentropy", metrics=["accuracy"])


dnn_3 = keras.Sequential([
    layers.Input(shape=(18,)),
    layers.Dense(256, activation="elu"),
    layers.Dense(128, activation="elu"),
    layers.Dense(64, activation="elu"),
    layers.Dense(32, activation="elu"),
    layers.Dense(1, activation="sigmoid")
], name="Model_3_Funnel_ELU_MSE")
dnn_3.compile(optimizer=keras.optimizers.RMSprop(learning_rate=1e-3),
              loss="mean_squared_error", metrics=["accuracy"])

ex3_models = {
    "DNN 1 (Shallow Tanh + SGD)": dnn_1,
    "DNN 2 (Deep BN/Dropout + Adam)": dnn_2,
    "DNN 3 (Funnel ELU + RMSprop/MSE)": dnn_3
}

val_aucs = {}
plt.figure(figsize=(8, 6))

for name, model in ex3_models.items():
    print(f"\n--- Training {name} ---")
    hist = model.fit(
        X_Train, y_Train,
        validation_data=(X_Val, y_Val),
        epochs=15,
        batch_size=1024,
        callbacks=[early_stop],
        verbose=1
    )
    plot_learning_curves(hist, model_name=name)
    
    val_preds = model.predict(X_Val, batch_size=2048).ravel()
    fpr, tpr, _ = roc_curve(y_Val, val_preds)
    roc_auc = auc(fpr, tpr)
    val_aucs[name] = roc_auc

for name, model in ex3_models.items():
    val_preds = model.predict(X_Val, batch_size=2048).ravel()
    fpr, tpr, _ = roc_curve(y_Val, val_preds)
    plt.plot(fpr, tpr, lw=2, label=f"{name} (Val AUC = {val_aucs[name]:.4f})")

plt.plot([0, 1], [0, 1], "k--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Exercise 3: Validation ROC Comparison of 3 DNN Architectures")
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.show()

best_dnn_name = max(val_aucs, key=val_aucs.get)
best_dnn_model = ex3_models[best_dnn_name]
print(f"Best DNN selected on Validation Sample: {best_dnn_name} (Val AUC = {val_aucs[best_dnn_name]:.4f})")

## Exercise 4

Repeat Exercise 4 from Lab 8 (parts b and c), adding your best performing DNN as one of the models. Choose your best DNN using the validation sample, then compare on the test sample.

In [ ]:

from IPython.display import HTML, display
import tabulate


plt.figure(figsize=(9, 7))


dnn_test_sets = [
    ("DNN - RawNames (8)", ex2_models["Raw Quantities (8)"], X_Test_Raw),
    ("DNN - FeatureNames (10)", ex2_models["Derived Features (10)"], X_Test_Feat),
    (f"Best DNN ({best_dnn_name}) - All 18", best_dnn_model, X_Test)
]

for label, model, x_te in dnn_test_sets:
    preds = model.predict(x_te, batch_size=2048).ravel()
    fpr, tpr, _ = roc_curve(y_Test, preds)
    plt.plot(fpr, tpr, lw=2, label=f"{label} (Test AUC = {auc(fpr, tpr):.4f})")


lda_clf = DA.LinearDiscriminantAnalysis().fit(X_Train, y_Train)
lda_preds = lda_clf.decision_function(X_Test)
fpr_lda, tpr_lda, _ = roc_curve(y_Test, lda_preds)
plt.plot(fpr_lda, tpr_lda, lw=2, linestyle="--", label=f"LDA All 18 (Test AUC = {auc(fpr_lda, tpr_lda):.4f})")

gb_clf = HistGradientBoostingClassifier(max_iter=100, random_state=42).fit(X_Train[:100000], y_Train[:100000])
gb_preds = gb_clf.predict_proba(X_Test)[:, 1]
fpr_gb, tpr_gb, _ = roc_curve(y_Test, gb_preds)
plt.plot(fpr_gb, tpr_gb, lw=2, linestyle=":", label=f"HistGBDT All 18 (Test AUC = {auc(fpr_gb, tpr_gb):.4f})")

plt.plot([0, 1], [0, 1], "k--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Lab 9 Exercise 4b: Test Sample ROC Comparison (DNN vs. Classical ML)")
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.show()


best_dnn_test_preds = best_dnn_model.predict(X_Test, batch_size=2048).ravel()
fpr_dnn, tpr_dnn, thresh_dnn = roc_curve(y_Test, best_dnn_test_preds)

scenarios = [(10, 100), (100, 1000), (1000, 10000), (10000, 100000)]
dnn_sig_table = []

for Ns, Nb in scenarios:
 
    Ns_p = tpr_dnn * Ns
    Nb_p = fpr_dnn * Nb
    denom = np.sqrt(Ns_p + Nb_p)
    sigma_dnn = np.divide(Ns_p, denom, out=np.zeros_like(Ns_p), where=denom > 0)
    best_i = np.argmax(sigma_dnn)
    
    
    sigma_lda = np.divide(tpr_lda * Ns, np.sqrt(tpr_lda * Ns + fpr_lda * Nb + 1e-12))
    
    dnn_sig_table.append([
        f"N_S={Ns}, N_B={Nb}",
        f"{thresh_dnn[best_i]:.4f}",
        f"{tpr_dnn[best_i]:.4f}",
        f"{fpr_dnn[best_i]:.4f}",
        f"{sigma_lda.max():.3f}",
        f"{sigma_dnn[best_i]:.3f}"
    ])

display(HTML(f"<h4>Lab 9 Exercise 4c: Test Sample Maximal Significance ({best_dnn_name})</h4>"))
display(HTML(tabulate.tabulate(
    dnn_sig_table,
    headers=["Scenario", "Optimal DNN Cut", "TPR (eps_S)", "FPR (eps_B)", "LDA Max Sig", "Best DNN Max Sig"],
    tablefmt="html"
)))